# Titanic Survival: Leakage-Safe Modeling and Interpretation

**Question.** How much out-of-sample predictive information is available, and which features contribute after preprocessing is learned only from training data?

We use stratified cross-validation, a preprocessing pipeline, calibrated metrics, permutation importance, and PCA as an exploratory visualization—not as a claim that latent components cause survival.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); df=pd.read_csv(ROOT/'titanic.csv')
y=df.pop('Survived'); X=df.drop(columns=['PassengerId','Name','Ticket','Cabin'])
cat=['Sex','Embarked']; num=[c for c in X.columns if c not in cat]
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models={'logistic':LogisticRegression(max_iter=2000,class_weight='balanced'),'random_forest':RandomForestClassifier(n_estimators=400,min_samples_leaf=3,random_state=42,class_weight='balanced')}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

## 1. Cross-validated predictive performance

Accuracy is reported alongside balanced accuracy, ROC-AUC, and average precision because the target is not perfectly balanced. Cross-validation estimates expected performance on similar unseen passengers; it does not estimate causal effects or historical counterfactuals.

In [ ]:
rows=[]
for name,model in models.items():
    pipe=Pipeline([('preprocess',pre),('model',model)])
    scores=cross_validate(pipe,X,y,cv=cv,scoring=['accuracy','balanced_accuracy','roc_auc','average_precision'])
    rows.append({'model':name,**{m.replace('test_',''):f'{scores[m].mean():.3f} ± {scores[m].std():.3f}' for m in scores if m.startswith('test_')}})
display(pd.DataFrame(rows).set_index('model'))

## 2. Held-out diagnostics and permutation importance

Permutation importance measures the decrease in held-out score after shuffling one feature. It is conditional on the model and can distribute importance among correlated predictors; it should not be read as a causal effect or universal feature ranking.

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
pipe=Pipeline([('preprocess',pre),('model',models['logistic'])]); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte)
print(classification_report(yte,pred,digits=3)); ConfusionMatrixDisplay.from_predictions(yte,pred,cmap='Blues'); plt.title('Held-out confusion matrix'); plt.show()
imp=permutation_importance(pipe,Xte,yte,n_repeats=30,random_state=42,scoring='roc_auc')
display(pd.DataFrame({'feature':X.columns,'importance':imp.importances_mean,'sd':imp.importances_std}).sort_values('importance',ascending=False))

## 3. Feature engineering and PCA as a descriptive lens

We create family size and an indicator for traveling alone. These are pre-outcome features. PCA is fit after imputation and scaling and is used only to visualize geometry; components maximize variance, not survival separation.

In [ ]:
X2=X.copy(); X2['FamilySize']=X2['SibSp']+X2['Parch']+1; X2['IsAlone']=(X2['FamilySize']==1).astype(int)
numeric=X2.select_dtypes(include='number'); Z=SimpleImputer(strategy='median').fit_transform(numeric); Z=StandardScaler().fit_transform(Z)
pca=PCA(n_components=2,random_state=42).fit_transform(Z); plot=pd.DataFrame({'PC1':pca[:,0],'PC2':pca[:,1],'Survived':y.map({0:'No',1:'Yes'})})
sns.scatterplot(data=plot,x='PC1',y='PC2',hue='Survived',alpha=.55); plt.title('PCA projection of numeric passenger features'); plt.show()

## Final modeling conclusions

* Leakage-safe pipelines keep imputation, scaling, and encoding inside each cross-validation fold.
* Predictive performance should be reported as a distribution across folds, not a single lucky split.
* Permutation importance indicates predictive reliance, not causal influence.
* Feature engineering can improve representation, but every feature must be available at prediction time and evaluated through the same cross-validation design.
* The dataset is historical and observational; model explanations should not be generalized to modern populations or treated as policy recommendations.